In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "SMQ_D.xpt"
csv_file = DATA_DIR / "SMQ_D_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Untouched source for final fidelity checking
source_df = df.copy()


print("=" * 80)
print("NHANES 2005-2006 SMQ_D XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "SMQ020",
    "SMD030",
    "SMQ040",
    "SMQ050Q",
    "SMQ050U",
    "SMD055",
    "SMD057",
    "SMD070",
    "SMD075",
    "SMQ077",
    "SMD641",
    "SMD650",
    "SMD093",
    "SMDUPCA",
    "SMD100BR",
    "SMD100FL",
    "SMD100MN",
    "SMD100LN",
    "SMD100TR",
    "SMD100NI",
    "SMD100CO",
    "SMQ620",
    "SMD630",
    "SMQ660",
    "SMQ664M",
    "SMQ664C",
    "SMQ664W",
    "SMQ664B",
    "SMQ664O",
    "SMQ666M",
    "SMQ666C",
    "SMQ666K",
    "SMQ666W",
    "SMQ666B",
    "SMQ666S",
    "SMQ666O",
    "SMQ670",
    "SMAQUEX2"
]


if len(df) != 7186:
    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 39:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "SMQ_D columns/order differ.\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: SMQ_D structure = "
    "7,186 rows x 39 columns."
)


# ============================================================
# 4. IDENTIFY NUMERIC / STRING / DECIMAL FIELDS
# ============================================================

STRING_COLUMNS = [
    "SMDUPCA",
    "SMD100BR"
]

DECIMAL_COLUMNS = [
    "SMD100NI"
]

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()

INTEGER_NUMERIC_COLUMNS = [
    col
    for col in numeric_cols
    if col not in DECIMAL_COLUMNS
]


print("\nString fields:")
print(STRING_COLUMNS)

print("\nGenuine decimal fields:")
print(DECIMAL_COLUMNS)


# ============================================================
# 5. VERIFY STRING FIELDS REALLY ARE STRINGS
# ============================================================

for col in STRING_COLUMNS:

    if not df[col].map(
        lambda x: isinstance(x, str)
    ).all():

        raise ValueError(
            f"{col} contains unexpected non-string values."
        )


print(
    "\nPASS: SMDUPCA and SMD100BR "
    "remain text fields."
)


# ============================================================
# 6. SMDUPCA FIDELITY CHECK
# ============================================================

#
# VERY IMPORTANT:
#
# SMDUPCA is NOT an integer.
#
# It is a character UPC field.
# Leading zeros are meaningful and must remain.
#

upc_nonblank = (
    df.loc[
        df["SMDUPCA"] != "",
        "SMDUPCA"
    ]
)


print("\n--- SMDUPCA SOURCE CHECK ---")

print(
    "Nonblank UPC values:",
    len(upc_nonblank)
)

print(
    "Leading-zero UPC values:",
    int(
        upc_nonblank
        .str.startswith("0")
        .sum()
    )
)

print(
    "'* NO MATCH *' values:",
    int(
        upc_nonblank
        .eq("* NO MATCH *")
        .sum()
    )
)


if len(upc_nonblank) != 757:
    raise ValueError(
        "Unexpected SMDUPCA nonblank count."
    )


if int(
    upc_nonblank
    .str.startswith("0")
    .sum()
) != 703:

    raise ValueError(
        "Unexpected number of leading-zero UPC values."
    )


if int(
    upc_nonblank
    .eq("* NO MATCH *")
    .sum()
) != 7:

    raise ValueError(
        "Unexpected '* NO MATCH *' UPC count."
    )


print(
    "PASS: SMDUPCA source text validated."
)


# ============================================================
# 7. KNOWN SAS/XPORT TINY-ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{int(true_zero_counts.sum()):,}"
)

print(
    "Tiny artifact occurrences:",
    f"{int(tiny_counts.sum()):,}"
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 8. EXPECTED VALIDATED TINY COUNTS
# ============================================================

EXPECTED_TINY_COUNTS = {

    "SMD030": 137,

    "SMD641": 216,

    "SMD100FL": 16,

    "SMD100MN": 745,

    "SMD100TR": 33
}


EXPECTED_TINY_TOTAL = 1147


# ============================================================
# 9. VALIDATE TINY ARTIFACT LOCATIONS
# ============================================================

actual_tiny_columns = set(
    tiny_counts[
        tiny_counts > 0
    ].index
)


expected_tiny_columns = set(
    EXPECTED_TINY_COUNTS.keys()
)


unexpected_tiny_columns = (
    actual_tiny_columns
    -
    expected_tiny_columns
)


if unexpected_tiny_columns:

    raise ValueError(
        "STOP: Tiny values found in unexpected "
        "SMQ_D variables:\n"
        + str(
            sorted(
                unexpected_tiny_columns
            )
        )
    )


missing_tiny_columns = (
    expected_tiny_columns
    -
    actual_tiny_columns
)


if missing_tiny_columns:

    raise ValueError(
        "Expected tiny-value variables missing:\n"
        + str(
            sorted(
                missing_tiny_columns
            )
        )
    )


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        tiny_counts[col]
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


if int(
    tiny_counts.sum()
) != EXPECTED_TINY_TOTAL:

    raise ValueError(
        "Unexpected total tiny-value count."
    )


print(
    "\nPASS: Exactly 1,147 validated "
    "tiny XPORT artifacts found."
)


# ============================================================
# 10. RESTORE ONLY VALIDATED ZEROS
# ============================================================

print("\n--- ZERO RESTORATION ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    df.loc[
        mask,
        col
    ] = 0.0


    print(
        f"{col:<10}: "
        f"{actual_count:,} tiny -> 0"
    )


# ============================================================
# 11. VERIFY RESTORED ZERO COUNTS
# ============================================================

print("\n--- RESTORED ZERO COUNTS ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    print(
        f"{col:<10}: "
        f"{zero_count:,}"
    )


    if zero_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} zeros, "
            f"found {zero_count:,}."
        )


print(
    "PASS: All 1,147 validated zeros restored."
)


# ============================================================
# 12. VERIFY NO TINY ARTIFACT REMAINS
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nRemaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny-value artifact remains."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 13. CDC CHECK - SMD030
# ============================================================

#
# Age started smoking regularly.
#
# CDC:
#
# 5-59 = 2,202
# 0 = Never smoked regularly = 137
# 999 = Don't know = 9
# Missing = 4,838
#

s = df["SMD030"]


if not (
    int(
        s.between(
            5,
            59
        ).sum()
    ) == 2202

    and

    int(
        s.eq(0)
        .sum()
    ) == 137

    and

    int(
        s.eq(999)
        .sum()
    ) == 9

    and

    int(
        s.isna()
        .sum()
    ) == 4838
):

    raise ValueError(
        "SMD030 differs from CDC."
    )


print(
    "\nPASS: SMD030 matches CDC."
)

print(
    "      Never smoked regularly = 137"
)


# ============================================================
# 14. CDC CHECK - SMD641
# ============================================================

#
# Days smoked cigarettes during past 30 days.
#
# CDC released range:
#
# 0-30 = 1,604
# 99 = Don't know = 3
# Missing = 5,579
#
# Attached XPT contains 216 zeros.
#

s = df["SMD641"]


if not (
    int(
        s.between(
            0,
            30
        ).sum()
    ) == 1604

    and

    int(
        s.eq(0)
        .sum()
    ) == 216

    and

    int(
        s.eq(99)
        .sum()
    ) == 3

    and

    int(
        s.isna()
        .sum()
    ) == 5579
):

    raise ValueError(
        "SMD641 differs from CDC/source."
    )


print(
    "PASS: SMD641 matches CDC/source."
)

print(
    "      Restored zeros = 216"
)


# ============================================================
# 15. CDC CHECK - SMD100FL
# ============================================================

#
# Filter type:
#
# 0 Non-filter = 16
# 1 Filter = 1,051
# Missing = 6,119
#

expected = {
    0: 16,
    1: 1051
}


actual = (
    df["SMD100FL"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != expected:

    raise ValueError(
        "SMD100FL frequencies differ."
    )


if int(
    df["SMD100FL"]
    .isna()
    .sum()
) != 6119:

    raise ValueError(
        "SMD100FL missing count differs."
    )


print(
    "PASS: SMD100FL matches CDC."
)


# ============================================================
# 16. CDC CHECK - SMD100MN
# ============================================================

#
# Menthol indicator:
#
# 0 Non-menthol = 745
# 1 Menthol = 322
# Missing = 6,119
#

expected = {
    0: 745,
    1: 322
}


actual = (
    df["SMD100MN"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != expected:

    raise ValueError(
        "SMD100MN frequencies differ."
    )


if int(
    df["SMD100MN"]
    .isna()
    .sum()
) != 6119:

    raise ValueError(
        "SMD100MN missing count differs."
    )


print(
    "PASS: SMD100MN matches CDC."
)


# ============================================================
# 17. CDC CHECK - SMD100TR
# ============================================================

#
# FTC Tar Content
#
# CDC released:
#
# 0-24 = 956
# Missing = 6,230
#
# Attached XPT has 33 valid zero values.
#

s = df["SMD100TR"]


if not (
    int(
        s.between(
            0,
            24
        ).sum()
    ) == 956

    and

    int(
        s.eq(0)
        .sum()
    ) == 33

    and

    int(
        s.isna()
        .sum()
    ) == 6230
):

    raise ValueError(
        "SMD100TR differs from CDC/source."
    )


print(
    "PASS: SMD100TR matches CDC/source."
)

print(
    "      Restored zero tar values = 33"
)


# ============================================================
# 18. CRITICAL DECIMAL CHECK - SMD100NI
# ============================================================

#
# SMD100NI = FTC Nicotine Content
#
# This is a REAL DECIMAL FIELD.
#
# CDC:
#
# 0.1-1.8 = 956
# Missing = 6,230
#
# DO NOT:
#
# - Int64 this variable
# - round it
# - recalculate it
# - remove legitimate ".0"
#

s = df["SMD100NI"]


if not (
    int(
        s.between(
            0.1,
            1.8
        ).sum()
    ) == 956

    and

    int(
        s.isna()
        .sum()
    ) == 6230
):

    raise ValueError(
        "SMD100NI differs from CDC."
    )


fractional_smd100ni = int(
    (
        ~np.isclose(
            s.dropna(),
            np.round(
                s.dropna()
            ),
            rtol=0,
            atol=1e-12
        )
    ).sum()
)


whole_smd100ni = int(
    (
        np.isclose(
            s.dropna(),
            np.round(
                s.dropna()
            ),
            rtol=0,
            atol=1e-12
        )
    ).sum()
)


print(
    "\n--- SMD100NI DECIMAL CHECK ---"
)

print(
    "Valid values:",
    int(
        s.notna()
        .sum()
    )
)

print(
    "Fractional values:",
    fractional_smd100ni
)

print(
    "Whole-number values:",
    whole_smd100ni
)


if fractional_smd100ni != 902:

    raise ValueError(
        "Unexpected SMD100NI fractional count."
    )


if whole_smd100ni != 54:

    raise ValueError(
        "Unexpected SMD100NI whole-value count."
    )


print(
    "PASS: SMD100NI is preserved as genuine decimal."
)


# ============================================================
# 19. VERIFY NO OTHER GENUINE DECIMAL VARIABLES
# ============================================================

print(
    "\n--- OTHER FRACTIONAL VALUE CHECK ---"
)


unexpected_fractional = {}


for col in INTEGER_NUMERIC_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional.sum()
    )


    if count > 0:

        unexpected_fractional[col] = count


if unexpected_fractional:

    raise ValueError(
        "STOP: Unexpected genuine decimal values found:\n"
        + str(
            unexpected_fractional
        )
    )


print(
    "PASS: SMD100NI is the ONLY genuine "
    "decimal numeric variable."
)


# ============================================================
# 20. SEQN VALIDATION
# ============================================================

print("\n--- SEQN CHECK ---")


seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy()
)


if not np.all(
    np.isclose(
        seqn_values,
        np.round(
            seqn_values
        ),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


seqn_missing = int(
    df["SEQN"]
    .isna()
    .sum()
)


seqn_duplicates = int(
    df["SEQN"]
    .duplicated()
    .sum()
)


seqn_min = df["SEQN"].min()
seqn_max = df["SEQN"].max()


print(
    "Missing:",
    seqn_missing
)

print(
    "Duplicates:",
    seqn_duplicates
)

print(
    "Minimum:",
    seqn_min
)

print(
    "Maximum:",
    seqn_max
)


if seqn_missing != 0:
    raise ValueError(
        "Unexpected missing SEQN."
    )


if seqn_duplicates != 0:
    raise ValueError(
        "Duplicate SEQN values found."
    )


if seqn_min != 31129:
    raise ValueError(
        f"Unexpected minimum SEQN: {seqn_min}"
    )


if seqn_max != 41474:
    raise ValueError(
        f"Unexpected maximum SEQN: {seqn_max}"
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 21. KEY CDC FREQUENCY CHECK - SMQ020
# ============================================================

EXPECTED_SMQ020 = {
    1: 2348,
    2: 2625,
    7: 3,
    9: 3
}


actual_smq020 = (
    df["SMQ020"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual_smq020 = {
    int(k): int(v)
    for k, v
    in actual_smq020.items()
}


if actual_smq020 != EXPECTED_SMQ020:

    raise ValueError(
        "SMQ020 frequency mismatch."
    )


if int(
    df["SMQ020"]
    .isna()
    .sum()
) != 2207:

    raise ValueError(
        "SMQ020 missing count differs."
    )


print(
    "\nPASS: SMQ020 matches CDC."
)


# ============================================================
# 22. KEY CDC FREQUENCY CHECK - SMQ040
# ============================================================

EXPECTED_SMQ040 = {
    1: 910,
    2: 179,
    3: 1259
}


actual_smq040 = (
    df["SMQ040"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual_smq040 = {
    int(k): int(v)
    for k, v
    in actual_smq040.items()
}


if actual_smq040 != EXPECTED_SMQ040:

    raise ValueError(
        "SMQ040 frequency mismatch."
    )


if int(
    df["SMQ040"]
    .isna()
    .sum()
) != 4838:

    raise ValueError(
        "SMQ040 missing count differs."
    )


print(
    "PASS: SMQ040 matches CDC."
)


# ============================================================
# 23. KEY CDC FREQUENCY CHECK - SMQ620
# ============================================================

EXPECTED_SMQ620 = {
    1: 765,
    2: 1249,
    7: 1,
    9: 3
}


actual_smq620 = (
    df["SMQ620"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual_smq620 = {
    int(k): int(v)
    for k, v
    in actual_smq620.items()
}


if actual_smq620 != EXPECTED_SMQ620:

    raise ValueError(
        "SMQ620 frequency mismatch."
    )


if int(
    df["SMQ620"]
    .isna()
    .sum()
) != 5168:

    raise ValueError(
        "SMQ620 missing count differs."
    )


print(
    "PASS: SMQ620 matches CDC."
)


# ============================================================
# 24. STRING-FIELD COUNTS
# ============================================================

if int(
    df["SMDUPCA"]
    .ne("")
    .sum()
) != 757:

    raise ValueError(
        "SMDUPCA nonblank count differs."
    )


if int(
    df["SMD100BR"]
    .ne("")
    .sum()
) != 1067:

    raise ValueError(
        "SMD100BR nonblank count differs."
    )


print(
    "PASS: SMDUPCA/SMD100BR "
    "nonblank counts match CDC."
)


# ============================================================
# 25. BUILD EXPECTED DATASET
# ============================================================

#
# expected_df =
#
# ORIGINAL XPT
#
# +
#
# ONLY these validated corrections:
#
# SMD030   tiny -> 0
# SMD641   tiny -> 0
# SMD100FL tiny -> 0
# SMD100MN tiny -> 0
# SMD100TR tiny -> 0
#
# SMD100NI is NOT modified.
#
# SMDUPCA is NOT modified.
#
# SMD100BR is NOT modified.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 26. CLEAN ONLY PROVEN WHOLE-NUMBER NUMERIC FIELDS
# ============================================================

#
# DO NOT run:
#
# for col in df.columns:
#     df[col] = df[col].astype("Int64")
#
# That would DAMAGE:
#
# SMD100NI decimal values
# SMDUPCA text / leading zeros
# SMD100BR text
#

for col in INTEGER_NUMERIC_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: Whole-number numeric fields "
    "converted to Int64."
)

print(
    "PASS: SMD100NI left as original float."
)

print(
    "PASS: SMDUPCA/SMD100BR left as text."
)


# ============================================================
# 27. PRE-EXPORT NUMERIC FIDELITY
# ============================================================

print(
    "\n--- PRE-EXPORT NUMERIC FIDELITY ---"
)


pre_export_failures = []


for col in numeric_cols:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    if col == "SMD100NI":

        current_values = (
            df[col]
            .to_numpy()
        )

    else:

        current_values = (
            df[col]
            .astype("Float64")
            .to_numpy(
                dtype=float,
                na_value=np.nan
            )
        )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Numeric values match expected XPT."
)


# ============================================================
# 28. PRE-EXPORT STRING FIDELITY
# ============================================================

for col in STRING_COLUMNS:

    if not (
        df[col]
        .to_numpy()
        ==
        expected_df[col]
        .to_numpy()
    ).all():

        raise ValueError(
            f"{col} text changed before export."
        )


print(
    "PASS: String values preserved exactly."
)


# ============================================================
# 29. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO:
#
# float_format
# round()
# numeric conversion of SMDUPCA
# recalculation
#
# SMD100NI remains the source-decoded float.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 30. RE-READ NUMERIC CSV DATA
# ============================================================

#
# Explicitly force the two string fields to strings.
#
# This prevents pandas from interpreting UPCs as numbers.
#

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip",
    dtype={
        "SMDUPCA": "string",
        "SMD100BR": "string"
    }
)


# ============================================================
# 31. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 32. EXACT NUMERIC XPT -> CSV FIDELITY
# ============================================================

print(
    "\n--- XPT -> CSV NUMERIC FIDELITY ---"
)


numeric_failures = []
total_numeric_differences = 0


for col in numeric_cols:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += (
        differences
    )


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Total numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All numeric CSV values "
    "match expected XPT values exactly."
)


# ============================================================
# 33. NUMERIC MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in numeric_cols:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


    missing_mismatches += (
        differences
    )


    if differences > 0:

        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "\nMissing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing positions changed:\n"
        + str(
            missing_failures
        )
    )


print(
    "PASS: Numeric missing positions "
    "preserved exactly."
)


# ============================================================
# 34. READ RAW CSV TOKENS
# ============================================================

#
# Raw token checking is especially important for:
#
# - SMDUPCA leading zeros
# - SMD100BR text
# - SMD100NI decimal representation
#

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)

    raw_rows = list(reader)


if len(raw_rows) != 7186:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 35. STRICT RAW CSV FIDELITY
# ============================================================

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        # ----------------------------------------------------
        # STRING FIELDS
        # ----------------------------------------------------

        if col in STRING_COLUMNS:

            expected_text = (
                source_value
            )


        # ----------------------------------------------------
        # GENUINE DECIMAL FIELD
        # ----------------------------------------------------

        elif col == "SMD100NI":

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(
                    float(source_value)
                )
            )


        # ----------------------------------------------------
        # WHOLE-NUMBER NUMERIC FIELDS
        # ----------------------------------------------------

        else:

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(
                    int(source_value)
                )
            )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )


    for example in raw_examples:

        print(
            example
        )


    raise ValueError(
        "SMQ_D raw CSV formatting differs."
    )


print(
    "PASS: Raw CSV cell formatting "
    "matches expected source representation."
)


# ============================================================
# 36. SMDUPCA RAW TEXT CHECK
# ============================================================

raw_upcs = [
    row["SMDUPCA"]
    for row in raw_rows
    if row["SMDUPCA"] != ""
]


raw_upc_leading_zero = sum(
    value.startswith("0")
    for value in raw_upcs
)


raw_no_match = sum(
    value == "* NO MATCH *"
    for value in raw_upcs
)


print(
    "\n--- SMDUPCA FINAL RAW CHECK ---"
)

print(
    "Nonblank UPCs:",
    len(raw_upcs)
)

print(
    "Leading-zero UPCs:",
    raw_upc_leading_zero
)

print(
    "'* NO MATCH *':",
    raw_no_match
)


if len(raw_upcs) != 757:

    raise ValueError(
        "UPC count changed."
    )


if raw_upc_leading_zero != 703:

    raise ValueError(
        "UPC leading zeros were damaged."
    )


if raw_no_match != 7:

    raise ValueError(
        "UPC '* NO MATCH *' values changed."
    )


print(
    "PASS: SMDUPCA preserved exactly, "
    "including leading zeros."
)


# ============================================================
# 37. UNWANTED .0 CHECK
# ============================================================

#
# DO NOT globally reject every ".0".
#
# SMD100NI is a genuine decimal field.
#
# Example:
#
# source 1.0 must remain "1.0"
#
# Only whole-number questionnaire/code fields
# must NOT end in ".0".
#

unwanted_dot_zero_count = 0
unwanted_dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col in INTEGER_NUMERIC_COLUMNS:

        value = row[col]


        if (
            value != ""
            and
            value.endswith(".0")
        ):

            unwanted_dot_zero_count += 1


            if len(
                unwanted_dot_zero_examples
            ) < 20:

                unwanted_dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nUnwanted integer-field '.0' values:",
    unwanted_dot_zero_count
)


if unwanted_dot_zero_count != 0:

    print(
        "Examples:",
        unwanted_dot_zero_examples
    )

    raise ValueError(
        "Unnecessary .0 remains in "
        "whole-number numeric fields."
    )


print(
    "PASS: No unwanted .0 in integer fields."
)


# ============================================================
# 38. LEGITIMATE SMD100NI .0 CHECK
# ============================================================

#
# In the attached XPT:
#
# SMD100NI = 1.0 occurs 54 times.
#
# Those should remain:
#
# 1.0
#
# NOT:
#
# 1
#

nicotine_dot_zero_count = sum(
    row["SMD100NI"].endswith(".0")
    for row in raw_rows
    if row["SMD100NI"] != ""
)


print(
    "Legitimate SMD100NI values ending '.0':",
    nicotine_dot_zero_count
)


if nicotine_dot_zero_count != 54:

    raise ValueError(
        "SMD100NI legitimate .0 formatting changed."
    )


print(
    "PASS: Genuine SMD100NI .0 values preserved."
)


# ============================================================
# 39. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = (
            value.lower()
        )


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        "Examples:",
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 40. FINAL TINY-ARTIFACT TEXT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings in CSV:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact remains in CSV."
    )


print(
    "PASS: No tiny artifacts in CSV."
)


# ============================================================
# 41. FINAL ZERO CHECK
# ============================================================

print(
    "\n--- FINAL CSV ZERO CHECK ---"
)


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        csv_df[col]
        .eq(0)
        .sum()
    )


    print(
        f"{col:<10}: "
        f"{actual_count:,}"
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: final zero count changed."
        )


print(
    "PASS: Final zero restorations remain correct."
)


# ============================================================
# 42. FINAL SMD100NI EXACT RAW CHECK
# ============================================================

nicotine_raw_mismatches = 0


for i, row in enumerate(
    raw_rows
):

    value = (
        expected_df.iloc[i][
            "SMD100NI"
        ]
    )


    expected_text = (
        ""
        if pd.isna(value)
        else str(
            float(value)
        )
    )


    if (
        row["SMD100NI"]
        !=
        expected_text
    ):

        nicotine_raw_mismatches += 1


print(
    "\nSMD100NI raw differences:",
    nicotine_raw_mismatches
)


if nicotine_raw_mismatches != 0:

    raise ValueError(
        "SMD100NI decimal fidelity changed."
    )


print(
    "PASS: SMD100NI decimal representation "
    "preserved exactly."
)


# ============================================================
# 43. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("SMQ_D CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


print(
    "\nValidated tiny -> zero:"
)

print(
    "SMD030:",
    137
)

print(
    "SMD641:",
    216
)

print(
    "SMD100FL:",
    16
)

print(
    "SMD100MN:",
    745
)

print(
    "SMD100TR:",
    33
)

print(
    "TOTAL:",
    EXPECTED_TINY_TOTAL
)


print(
    "\nGenuine decimal fields:"
)

print(
    "SMD100NI"
)

print(
    "SMD100NI fractional values:",
    fractional_smd100ni
)

print(
    "SMD100NI legitimate .0 values:",
    nicotine_dot_zero_count
)


print(
    "\nString fields preserved:"
)

print(
    "SMDUPCA"
)

print(
    "SMD100BR"
)

print(
    "SMDUPCA leading-zero values:",
    raw_upc_leading_zero
)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Unwanted integer .0 values:",
    unwanted_dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "SMQ_D_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2005-2006 SMQ_D XPT -> CSV
Rows: 7,186
Columns: 39
PASS: SMQ_D structure = 7,186 rows x 39 columns.

String fields:
['SMDUPCA', 'SMD100BR']

Genuine decimal fields:
['SMD100NI']

PASS: SMDUPCA and SMD100BR remain text fields.

--- SMDUPCA SOURCE CHECK ---
Nonblank UPC values: 757
Leading-zero UPC values: 703
'* NO MATCH *' values: 7
PASS: SMDUPCA source text validated.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 1,147

Tiny values by variable:
SMD100MN    745
SMD641      216
SMD030      137
SMD100TR     33
SMD100FL     16

PASS: Exactly 1,147 validated tiny XPORT artifacts found.

--- ZERO RESTORATION ---
SMD030    : 137 tiny -> 0
SMD641    : 216 tiny -> 0
SMD100FL  : 16 tiny -> 0
SMD100MN  : 745 tiny -> 0
SMD100TR  : 33 tiny -> 0

--- RESTORED ZERO COUNTS ---
SMD030    : 137
SMD641    : 216
SMD100FL  : 16
SMD100MN  : 745
SMD100TR  : 33
PASS: All 1,147 validated zeros restored.

Remaining tiny artifacts: 0
PASS: No tiny-value artifacts remai